# Claude playground — prueba aislada, no toca `app/`

Objetivo mínimo (alcance recortado por Erik el 2026-08-29): consultar los índices reales de Azure AI Search (igual que hace `azure_tools.py`/`query_service.py`), traer el prompt real de AGB (canal `website`, desde `app/config/tenants/agb/prompts/`), y correr unas preguntas de prueba contra Claude.

Sin tool-calling, sin manejo de errores, sin load balancer todavía — solo pregunta → retrieval manual → respuesta de Claude.

**Vía Microsoft Foundry** (no la API directa de Anthropic) — mismo patrón de infraestructura que ya usan para Azure OpenAI, sin depender de residencia de datos de EE.UU. de la API directa (ver `agb_claude_migration` en memoria para el porqué). Requiere `AZURE_FOUNDRY_CLAUDE_API_KEY` en `.env`.

In [1]:
import os
import sys
import json
import requests
from dotenv import load_dotenv

# repo root en sys.path (notebook vive en testing/notebooks/)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

load_dotenv(os.path.join(REPO_ROOT, ".env"))

AZURE_AI_SEARCH_ENDPOINT = os.environ["AZURE_AI_SEARCH_ENDPOINT"]
AZURE_AI_SEARCH_API_KEY = os.environ["AZURE_AI_SEARCH_API_KEY"]
AZURE_AI_SEARCH_INDEX = os.environ["AZURE_AI_SEARCH_INDEX"]
AZURE_AI_SEARCH_PRICE_LIST_INDEX = os.environ["AZURE_AI_SEARCH_PRICE_LIST_INDEX"]
SEMANTIC_CONFIGURATION = os.environ["SEMANTIC_CONFIGURATION"]

# Claude vía Microsoft Foundry (mismo patrón de infra que ya usan para Azure OpenAI).
# Nota: el endpoint que muestra la consola de Foundry trae el sufijo OpenAI-compatible
# (".../openai/v1") -- Claude no habla ese dialecto, habla el Anthropic Messages API
# nativo en "/anthropic". Se reemplaza el sufijo aquí. Verificado con una llamada real.
AZURE_FOUNDRY_CLAUDE_API_KEY = os.environ["AZURE_FOUNDRY_CLAUDE_API_KEY"]
AZURE_FOUNDRY_CLAUDE_ENDPOINT = "https://foundry-test-clinyq-resource.openai.azure.com/anthropic"
AZURE_FOUNDRY_CLAUDE_DEPLOYMENT = "claude-sonnet-5-clinyq"

## 1. Prompt real de AGB (canal website)

Usa el mismo `FilesystemPromptConfigRepository` que ya existe en la migración hexagonal (`app/adapters/outbound/tenant_config/`) — así el notebook prueba el prompt que de verdad se usaría en producción, no una copia pegada a mano.

In [2]:
from app.adapters.outbound.tenant_config.filesystem_prompt_config_repository import FilesystemPromptConfigRepository

CONFIG_DIR = os.path.join(REPO_ROOT, "app", "config", "tenants")
prompt_repo = FilesystemPromptConfigRepository(config_dir=CONFIG_DIR)

base_prompt = await prompt_repo.get_base_prompt("agb", "website")
system_prompt = base_prompt.format(reply_language="español")

print(f"Prompt cargado: {len(system_prompt)} caracteres")
print(system_prompt[:500])

Prompt cargado: 28839 caracteres
Asume el ROL de un asistente virtual multilingüe de la clinica Antiaging Group Barcelona, una clínica médica especializada en cirugía estética.
Tu función es ofrecer información de manera amigable, clara, coherente, profesional y accesible a pacientes potenciales y actuales.
Resolver dudas generales sobre los servicios de la clínica y orientar a las usuarios hacia alguna de las opciones de valoracion.

REGLAS GENERALES:
- 1. Nunca asumas intención si no está explícita.
- 2. Responde SOLO con la 


## 2. Retrieval manual contra Azure AI Search

Claude no tiene equivalente al "on your data" de Azure OpenAI (grounding server-side) — hay que traer el contexto a mano y meterlo en el mensaje. Dos funciones, calcadas de las que ya existen en `app/services/cloud/azure/`:

- `search_price_list`: mismo payload que `azure_tools.py::procedures_and_treatments_price_list` (índice de precios, `searchMode="all"`).
- `search_main_index`: mismo payload que `query_service.py::AzureSearchQueryService.search_unstructured` (índice principal, búsqueda híbrida semántica + vectorial).

In [3]:
import re
import unicodedata

# En producción el LLM extrae la palabra clave vía tool-calling antes de llamar a este
# endpoint (que usa searchMode="all" -- exige que TODAS las palabras coincidan). Como
# aquí no hay tool-calling, se hace una limpieza mínima de stopwords para no mandar la
# pregunta completa tal cual (eso nunca matchea nada, ver CLAUDE.md "Price-list search
# relevance bug" -- el mismo problema existe hoy en producción, solo que oculto porque
# el LLM ya manda un argumento limpio).
_STOPWORDS = {
    "que", "cuanto", "cuanta", "cuesta", "cuestan", "es", "el", "la", "los", "las",
    "un", "una", "unos", "unas", "de", "del", "para", "por", "quiero", "quisiera",
    "me", "interesa", "gustaria", "arreglarme", "hacer", "hacerme", "operarme", "y",
    "en", "mi", "tu", "su", "antes", "pero", "no", "resultado", "con", "cuanto",
}


def _keywords(text: str) -> str:
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    words = [w for w in text.split() if w not in _STOPWORDS]
    return " ".join(words)


def search_price_list(query: str) -> dict:
    url = f"{AZURE_AI_SEARCH_ENDPOINT}/indexes/{AZURE_AI_SEARCH_PRICE_LIST_INDEX}/docs/search?api-version=2025-11-01-preview"
    headers = {"Content-Type": "application/json", "api-key": AZURE_AI_SEARCH_API_KEY}
    payload = {"search": _keywords(query), "searchMode": "all", "count": True}

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    return response.json().get("value", [])


def search_main_index(query: str, top_k: int = 5) -> list[str]:
    url = f"{AZURE_AI_SEARCH_ENDPOINT}/indexes/{AZURE_AI_SEARCH_INDEX}/docs/search?api-version=2025-11-01-preview"
    headers = {"Content-Type": "application/json", "api-key": AZURE_AI_SEARCH_API_KEY}
    payload = {
        "search": query,
        "top": top_k,
        "queryType": "semantic",
        "semanticConfiguration": SEMANTIC_CONFIGURATION,
        "captions": "extractive",
        "vectorQueries": [{"kind": "text", "text": query, "fields": "text_vector", "k": top_k}],
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    docs = response.json().get("value", [])

    chunks = []
    for doc in docs:
        captions = doc.get("@search.captions", [])
        if captions:
            chunks.extend(c["text"] for c in captions if c.get("text"))
        elif doc.get("chunk"):
            chunks.append(doc["chunk"])
    return chunks

## 3. Llamada a Claude

El contexto recuperado se inyecta como texto plano antes de la pregunta del usuario, dentro del mismo mensaje `user` — es el reemplazo manual más simple del grounding automático de Azure.

In [4]:
from anthropic import AnthropicFoundry

claude = AnthropicFoundry(
    api_key=AZURE_FOUNDRY_CLAUDE_API_KEY,
    base_url=AZURE_FOUNDRY_CLAUDE_ENDPOINT,
)


def ask_claude(question: str, model: str = AZURE_FOUNDRY_CLAUDE_DEPLOYMENT) -> dict:
    price_docs = search_price_list(question)
    chunks = search_main_index(question)

    context_parts = []
    if price_docs:
        context_parts.append("PRECIOS ENCONTRADOS:\n" + json.dumps(price_docs, ensure_ascii=False, indent=2))
    if chunks:
        context_parts.append("FRAGMENTOS RELEVANTES:\n" + "\n---\n".join(chunks))

    context_block = "\n\n".join(context_parts) if context_parts else "(sin resultados de búsqueda)"

    user_message = f"CONTEXTO RECUPERADO:\n{context_block}\n\nPREGUNTA DEL PACIENTE:\n{question}"

    response = claude.messages.create(
        model=model,
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )

    # el modelo viene con "extended thinking" por defecto -- content[] puede traer
    # un ThinkingBlock antes del TextBlock real, así que hay que buscarlo por tipo.
    answer = next((block.text for block in response.content if block.type == "text"), "")

    return {
        "question": question,
        "answer": answer,
        "price_docs_found": len(price_docs),
        "chunks_found": len(chunks),
    }

## 4. Preguntas de prueba

Un par de casos ya conocidos de sesiones de tuning anteriores (ver `CLAUDE.md`, sección "Prompt-tuning session status").

In [5]:
test_questions = [
    "¿cuánto cuesta una rinoplastia?",
    "quiero arreglarme los labios",
    "me interesa la lipo",
    "ya me operé el pecho antes pero no quedé contenta con el resultado, cuánto cuesta arreglarlo",
]

for q in test_questions:
    result = ask_claude(q)
    print(f"\n{'=' * 80}\nPREGUNTA: {result['question']}")
    print(f"(precios encontrados: {result['price_docs_found']}, fragmentos: {result['chunks_found']})\n")
    print(result["answer"])


PREGUNTA: ¿cuánto cuesta una rinoplastia?
(precios encontrados: 1, fragmentos: 5)

El precio de la **Rinoseptoplastia** (cirugía de nariz) es de **7.500€ - 8.500€**, e incluye honorarios médicos y de la clínica. El especialista a cargo de este procedimiento es el **Dr. Benito**.

Si deseas más información o una valoración personalizada, con gusto puedo orientarte sobre los siguientes pasos. 😊



PREGUNTA: quiero arreglarme los labios
(precios encontrados: 2, fragmentos: 5)

¿Te refieres a los labios de la boca o a la zona íntima?



PREGUNTA: me interesa la lipo
(precios encontrados: 0, fragmentos: 5)

No hay problema, con gusto te oriento. ¿En qué zona te gustaría hacer la liposucción: abdomen, flancos, brazos, espalda, piernas o papada?

Cuéntame la zona que te interesa y te doy la información correspondiente.



PREGUNTA: ya me operé el pecho antes pero no quedé contenta con el resultado, cuánto cuesta arreglarlo
(precios encontrados: 0, fragmentos: 5)

Entiendo que no haya sido el resultado que esperabas, y siento que estés pasando por esta situación. 😊

Como se trata de una revisión sobre una cirugía previa, el precio no es el mismo que el de una primera intervención, ya que este tipo de casos requiere valorar primero qué se realizó anteriormente y qué es lo que te gustaría corregir o mejorar ahora.

Por ello, voy a derivar tu caso para que un especialista lo revise y pueda confirmarte el enfoque y el coste correspondiente.

¿Podrías confirmarme tu nombre, correo y teléfono para que puedan contactarte a la mayor brevedad?
